In [0]:

# Databricks notebook source
# ============================================================
# BanglaPay Lakehouse Platform
# DAB Deployment Smoke Test
# ============================================================

# COMMAND ----------

from datetime import datetime, timezone
import json

# COMMAND ----------
# 1. Notebook parameters
# These values will later be supplied by the DAB Job.
# Local/default notebook execution uses DEV values.

def ensure_widget(name: str, default: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default)

ensure_widget("environment", "dev")
ensure_widget("workspace_host", "")
ensure_widget("data_volume_percent", "20")

environment = dbutils.widgets.get("environment").strip().lower()
workspace_host = dbutils.widgets.get("workspace_host").strip()
data_volume_percent = dbutils.widgets.get("data_volume_percent").strip()

print("========== BANGLAPAY DAB SMOKE TEST ==========")

# COMMAND ----------
# 2. Validate deployment parameters

if environment not in ("dev", "prod"):
    raise ValueError(f"Unsupported environment: {environment}")

expected_volume = {
    "dev": "20",
    "prod": "100",
}

if data_volume_percent != expected_volume[environment]:
    raise ValueError(
        f"Incorrect data volume for {environment}: "
        f"expected {expected_volume[environment]}%, "
        f"received {data_volume_percent}%"
    )

if workspace_host and not workspace_host.startswith("https://"):
    raise ValueError("Invalid workspace URL.")

print(f"Environment       : {environment.upper()}")
print(f"Workspace         : {workspace_host or 'Notebook UI execution'}")
print(f"Data volume policy: {data_volume_percent}%")

# COMMAND ----------
# 3. Validate Python runtime

python_version = __import__("sys").version.split()[0]

print(f"Python version    : {python_version}")

# COMMAND ----------
# 4. Validate Spark runtime

spark_version = spark.version

print(f"Spark version     : {spark_version}")

# COMMAND ----------
# 5. Execute a harmless Spark SQL test

result = spark.sql("""
    SELECT
        2 + 2 AS arithmetic_result,
        current_timestamp() AS execution_timestamp,
        current_catalog() AS current_catalog,
        current_schema() AS current_schema
""").first()

if result["arithmetic_result"] != 4:
    raise RuntimeError("Spark SQL smoke test failed.")

print("Spark SQL test    : PASSED")
print(f"Current catalog   : {result['current_catalog']}")
print(f"Current schema    : {result['current_schema']}")

# COMMAND ----------
# 6. Generate execution report

report = {
    "project": "banglapay-lakehouse-platform",
    "test_name": "dab_smoke_test",
    "status": "SUCCESS",
    "environment": environment,
    "workspace_host": workspace_host,
    "data_volume_percent": int(data_volume_percent),
    "python_version": python_version,
    "spark_version": spark_version,
    "arithmetic_result": result["arithmetic_result"],
    "current_catalog": result["current_catalog"],
    "current_schema": result["current_schema"],
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

print("\n========== SMOKE TEST REPORT ==========")
print(json.dumps(report, indent=2, default=str))

# COMMAND ----------

print("\nBANGLAPAY DAB SMOKE TEST: PASSED")